##### 1. Setup paths and config

In [ ]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"  # no new __pycache__

import warnings
warnings.filterwarnings("ignore")

import sys
sys.dont_write_bytecode = True  # prevent __pycache__ when running
import shutil
from pathlib import Path

# Remove any existing __pycache__ dirs
for root, dirs, _ in os.walk(Path.cwd(), topdown=False):
    for d in dirs:
        if d == "__pycache__":
            shutil.rmtree(Path(root) / d, ignore_errors=True)

# Run from repo root (Thesis-JP) or from fvqa/
nb_dir = Path.cwd()
fvqa_dir = nb_dir if (nb_dir / "models").exists() else nb_dir / "fvqa"
if not (fvqa_dir / "models").exists():
    raise FileNotFoundError("Run this notebook from project root or from fvqa/")
sys.path.insert(0, str(fvqa_dir))

from utils import load_config

config_path = fvqa_dir / "configs" / "default.yaml"
config = load_config(str(config_path)) if config_path.exists() else {}
data_cfg = config.get("data", {})
train_cfg = config.get("training", {})
paths_cfg = config.get("paths", {})

dataset_name = data_cfg.get("dataset_name", "HuggingFaceM4/VQAv2")
train_split = data_cfg.get("train_split", "train[:5000]")
batch_size = data_cfg.get("batch_size", 4)
num_workers = data_cfg.get("num_workers", 0)
image_size = data_cfg.get("image_size", 224)
max_length = data_cfg.get("max_length", 256)
model_cfg = config.get("model", {})
llm_name = model_cfg.get("llm_name", "TinyLlama/TinyLlama-1.1B-Chat-v1.0")
projection_hidden_dim = model_cfg.get("projection_hidden_dim", 0)
lr = train_cfg.get("learning_rate", 1e-4)
weight_decay = train_cfg.get("weight_decay", 0.01)
projection_dropout = train_cfg.get("projection_dropout", 0.0)
epochs = train_cfg.get("epochs", 5)
log_every = train_cfg.get("log_every", 10)
save_every = train_cfg.get("save_every", 1)
base_checkpoint_dir = fvqa_dir / paths_cfg.get("checkpoint_dir", "checkpoints")
projection_type = model_cfg.get("projection_type", "linear").lower()
checkpoint_dir = base_checkpoint_dir / projection_type
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print(f"Config loaded. Train split: {train_split}, batch_size: {batch_size}, epochs: {epochs}, projector: {projection_type}")

##### 2. Load frozen LLM and tokenizer

In [ ]:
from models.frozen_llm import load_frozen_llm

print("Loading tokenizer and frozen LLM...")
llm, tokenizer = load_frozen_llm(llm_name)
llm.eval()
main_device = next(llm.parameters()).device
print(f"LLM on {main_device}")

##### 3. Load vision encoder, projector, and FrozenVQA model

In [ ]:
from models.vision_encoder import VisionEncoder
from models.fvqa_model import LinearVisionToLLM, FrozenVQA
from models.query_transformer import QueryTransformer

vision_encoder = VisionEncoder()
vision_dim = model_cfg.get("vision_dim", 768)
llm_hidden = llm.config.hidden_size
if projection_type == "qformer":
    num_query_tokens = model_cfg.get("qformer_num_query_tokens", 16)
    num_layers = model_cfg.get("qformer_num_layers", 2)
    num_heads = model_cfg.get("qformer_num_heads", 8)
    qformer_dropout = model_cfg.get("qformer_dropout", projection_dropout)
    projector = QueryTransformer(vision_dim=vision_dim, llm_dim=llm_hidden, num_query_tokens=num_query_tokens, num_layers=num_layers, num_heads=num_heads, dropout=qformer_dropout)
else:
    projector = LinearVisionToLLM(vision_dim=vision_dim, llm_dim=llm_hidden, dropout=projection_dropout, hidden_dim=projection_hidden_dim)
model = FrozenVQA(vision_encoder=vision_encoder, projector=projector, llm=llm)

vision_encoder = vision_encoder.to(main_device)
projector = projector.to(main_device)

n_trainable = sum(p.numel() for p in projector.parameters())
print(f"Trainable parameters (projector only): {n_trainable:,}")

##### 4. Load VQA dataset and DataLoader

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from data.vqa_dataset import load_vqa_subset, VQADataset

print("Loading VQA subset...")
hf_split = train_split if "[" in train_split else f"{train_split}[:5000]"
hf_data = load_vqa_subset(dataset_name=dataset_name, split=hf_split)

image_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class TransformDataset(VQADataset):
    def __getitem__(self, idx):
        out = super().__getitem__(idx)
        out["image"] = image_transform(out["image"].convert("RGB"))
        return out

train_dataset = TransformDataset(
    hf_data,
    tokenizer=tokenizer,
    image_size=image_size,
    max_length=max_length,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

print(f"Dataset size: {len(train_dataset)}, batches: {len(train_loader)}")

##### 5. Training loop (projector only)

In [ ]:
from utils import save_checkpoint

# Prepare validation loader for per-epoch validation loss (reuse val_dataset if already created, else build once)
_val_loader = None
def get_val_loader():
    global _val_loader
    if _val_loader is None:
        from data.vqa_dataset import load_vqa_subset, VQADataset
        val_split = data_cfg.get("val_split", "validation[:1000]")
        val_data = load_vqa_subset(dataset_name=dataset_name, split=val_split)
        class ValTransformDataset(VQADataset):
            def __getitem__(self, idx):
                out = super().__getitem__(idx)
                out["image"] = image_transform(out["image"].convert("RGB"))
                return out
        val_dataset = ValTransformDataset(val_data, tokenizer=tokenizer, image_size=image_size, max_length=max_length)
        _val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return _val_loader

optimizer = torch.optim.AdamW(projector.parameters(), lr=lr, weight_decay=weight_decay)
loss_csv = checkpoint_dir / "loss_history.csv"
last_ckpt = checkpoint_dir / "projector_last.pt"

if not last_ckpt.exists():
    with open(loss_csv, "w") as f:
        f.write("epoch,train_loss,val_loss\n")

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(train_loader):
            images = batch["image"].to(main_device)
            input_ids = batch["input_ids"].to(main_device)
            attention_mask = batch["attention_mask"].to(main_device)
            labels = batch["labels"].to(main_device)

            outputs = model(
                image=images,
                question_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (step + 1) % log_every == 0:
                print(f"Epoch {epoch + 1} step {step + 1} loss={loss.item():.4f}")

        avg_loss = total_loss / max(len(train_loader), 1)
        print(f"Epoch {epoch + 1} avg_loss={avg_loss:.4f}")

        # Validation loss this epoch
        model.eval()
        val_loader = get_val_loader()
        total_val, n_val = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(main_device)
                input_ids = batch["input_ids"].to(main_device)
                attention_mask = batch["attention_mask"].to(main_device)
                labels = batch["labels"].to(main_device)
                outputs = model(image=images, question_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_val += outputs.loss.item() * images.size(0)
                n_val += images.size(0)
        val_loss = total_val / max(n_val, 1)
        print(f"Epoch {epoch + 1} val_loss={val_loss:.4f}")

        with open(loss_csv, "a") as f:
            f.write(f"{epoch + 1},{avg_loss:.6f},{val_loss:.6f}\n")

        if (epoch + 1) % save_every == 0:
            path = checkpoint_dir / f"projector_epoch_{epoch + 1}.pt"
            save_checkpoint(projector, optimizer, epoch + 1, str(path))
            print(f"Saved {path}")

    save_checkpoint(projector, optimizer, epochs, str(last_ckpt))
    print("Training done.")
else:
    print("Checkpoint found, skipping training.")

In [ ]:
from utils import plot_training_comparison

# Comparative plot: training and validation loss by projector type (linear, qformer, ...)
plot_training_comparison(base_checkpoint_dir)

##### 6. Optional: generation sample

In [ ]:
model.eval()
sample = train_dataset[5]
img = sample["image"].unsqueeze(0).to(main_device)

import matplotlib.pyplot as plt
import numpy as np
_mean = np.array([0.485, 0.456, 0.406])
_std = np.array([0.229, 0.224, 0.225])
_vis = sample["image"].permute(1, 2, 0).numpy()
_vis = np.clip(_std * _vis + _mean, 0, 1)
plt.figure(figsize=(5, 5))
plt.imshow(_vis)
plt.axis("off")
#plt.title("Image for generation sample")
plt.show()

prompt = "Question: describe this picture"
tok = tokenizer(prompt, return_tensors="pt").to(main_device)
input_ids, attn = tok["input_ids"], tok["attention_mask"]

with torch.no_grad():
    vision_feat = model.vision(img)
    vision_emb = model.projector(vision_feat)
    inputs_embeds = model.llm.get_input_embeddings()(input_ids)
    inputs_embeds[:, 0, :] = vision_emb
    out = model.llm.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attn,
        max_new_tokens=20,
        do_sample=False,
    )

generated = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"Generated: {generated}")

##### 7. Task Performance Metrics (EM, BLEU-1/4, CIDEr, ROUGE-L)

In [ ]:
from data.vqa_dataset import load_vqa_subset, VQADataset
from metrics import compute_all_metrics, print_metrics_report

# Eval subset size (smaller = faster)
metrics_split = data_cfg.get("val_split", "validation[:1000]")
metrics_n = 200  # cap for quick run in notebook; increase for full report
max_new_tokens = 30

print("Loading eval subset for metrics...")
eval_data = load_vqa_subset(dataset_name=dataset_name, split=metrics_split)

class EvalMetricsDataset(VQADataset):
    def __getitem__(self, idx):
        item = self.data[idx]
        q = item.get("question", "")
        a = item.get("multiple_choice_answer", item.get("answers", [""])[0] if isinstance(item.get("answers"), list) else str(item.get("answer", "")))
        if isinstance(a, list):
            a = a[0] if a else ""
        out = super().__getitem__(idx)
        out["image"] = image_transform(out["image"].convert("RGB"))
        out["question"], out["answer"] = q, a
        return out

eval_dataset = EvalMetricsDataset(eval_data, tokenizer=tokenizer, image_size=image_size, max_length=max_length)
eval_loader = DataLoader(eval_dataset, batch_size=1, shuffle=False)
n_eval = min(metrics_n, len(eval_dataset))
all_refs, all_preds = [], []

model.eval()
prompt_prefix, prompt_suffix = "Question: ", " Answer:"
with torch.no_grad():
    for idx, batch in enumerate(eval_loader):
        if idx >= n_eval:
            break
        img = batch["image"].to(main_device)
        q, ref = batch["question"][0], batch["answer"][0]
        vision_feat = model.vision(img)
        vision_emb = model.projector(vision_feat)
        prompt = prompt_prefix + q + prompt_suffix
        tok = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).to(main_device)
        input_ids, attn = tok["input_ids"], tok["attention_mask"]
        inputs_embeds = model.llm.get_input_embeddings()(input_ids)
        inputs_embeds[:, 0, :] = vision_emb
        out = model.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attn,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
        pred = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        all_refs.append(ref)
        all_preds.append(pred)
        if (idx + 1) % 50 == 0:
            print(f"  Generated {idx + 1}/{n_eval}")

metrics = compute_all_metrics(all_refs, all_preds)
print_metrics_report(metrics, title="Task Performance Metrics Report")